# Chapter 30
## The PING Model of Gamma Rhythms
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter30.ipynb)

## About this chapter

PING (pyramidal-interneuron gamma) is the canonical mechanism for gamma-band
rhythms in cortical networks: a population of excitatory (E) cells recruits
a population of inhibitory (I) cells, and the resulting recurrent inhibition
silences the E cells until it decays away and the next cycle can start. The
examples below build this mechanism up from a single E-I pair to sparse and
random populations of hundreds of cells, reading out E/I spike rasters and an
LFP-like mean-voltage trace.

E cells excite I cells and I cells inhibit E cells through synaptic
conductances such as $I_{\rm IE}=g_{\rm IE}s_I(E_I-v_E)$ and
$I_{\rm EI}=g_{\rm EI}s_E(E_E-v_I)$. Drive strength, E/I heterogeneity, and
recurrent E/I connectivity together decide which cells participate in each
cycle and how tightly the population synchronizes. Random-network synaptic
weights are normalized by the expected in-degree ($g_{\rm hat}/(N p)$), so
sparse and dense networks with the same mean input can be compared directly.

See [`chapter30.md`](chapter30.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

## RTM/WB neuron model and shared network helpers

Every example in this chapter uses the same two point-neuron models: an
excitatory Reduced-Traub-Miles (RTM) cell and an inhibitory Wang-Buzsaki (WB)
cell, coupled through synapses with a rise time and a decay time. `tau_peak_function`
and `tau_d_q_function` solve for the internal rise-time constant `tau_dq`
that makes the double-exponential synapse peak at the nominal `tau_peak`.
`rtm_init_population` / `wb_init_population` splay-initialize a whole
population of uncoupled cells around their limit cycle (or fixed point, if
below firing threshold) at prescribed phases -- this is what lets a
population simulation start already "running" instead of from a synchronized
transient.

`_ping_step_loop` is the shared, numba-accelerated core of every population
simulation in this chapter (PING_1 through PING_9): one Heun/midpoint step of
the full E/I network, with an inner loop over neurons and synapses so numba
can compile it to fast machine code. It is wrapped by `simulate_ping_network`,
which builds initial conditions and returns spike times/indices and an
LFP-like trace. Following this repo's numba convention, the per-step update
uses `math.pow(x, 3.0)`/`math.pow(x, 4.0)` instead of `x ** 3`/`x ** 4`
(numba's `**` differs from CPython's by about 1 ULP, which would otherwise
compound over a long integration), and the function is *not* decorated with
`@njit(cache=True)` (caching breaks under the `<dynamic>` module-name exec
pattern the tests use to load notebook definitions).

In [ ]:
import math

import numpy as np
from numpy import exp, tanh
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from numba import njit
from numba.typed import List
from ipywidgets import interact


# ------------------------------------------------------------- E cell (RTM)


def m_e_inf(v):
    alpha_m = 0.32 * (v + 54) / (1 - exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


def h_e_inf(v):
    alpha_h = 0.128 * exp(-(v + 50) / 18)
    beta_h = 4. / (1 + exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


def tau_h_e(v):
    alpha_h = 0.128 * exp(-(v + 50) / 18)
    beta_h = 4. / (1 + exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


def n_e_inf(v):
    alpha_n = 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))
    beta_n = 0.5 * exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


def tau_n_e(v):
    alpha_n = 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))
    beta_n = 0.5 * exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


# ------------------------------------------------------------- I cell (WB)


def m_i_inf(v):
    alpha_m = 0.1 * (v + 35) / (1 - exp(-(v + 35) / 10))
    beta_m = 4. * exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


def h_i_inf(v):
    alpha_h = 0.07 * exp(-(v + 58) / 20)
    beta_h = 1. / (exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


def tau_h_i(v):
    alpha_h = 0.07 * exp(-(v + 58) / 20)
    beta_h = 1. / (exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5


def n_i_inf(v):
    alpha_n = -0.01 * (v + 34) / (exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


def tau_n_i(v):
    alpha_n = -0.01 * (v + 34) / (exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5


# --------------------------------------------------------- double-exp synapse


def tau_peak_function(tau_d, tau_r, tau_d_q):
    dt_ = 0.01
    dt05_ = dt_ / 2
    s, t = 0., 0.
    s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05_ * s_inc
        s_inc_tmp = exp(-(t + dt05_) / tau_d_q) * (1 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt_ * s_inc_tmp
        t = t + dt_
        s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


def tau_d_q_function(tau_d, tau_r, tau_hat):
    tau_d_q_left = 1.
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2


# ------------------------------------------------------- population splay init


def rtm_init_population(i_ext, phi_vec):
    '''vectorized rtm_init over a population: each of len(i_ext) RTM
    neurons is integrated (Heun/midpoint) independently until its 3rd
    spike, then (v,h,n) is interpolated at phase phi_vec[i] between the
    2nd and 3rd spikes. Faithfully reproduces the matlab source\'s bug:
    m_tmp is computed from the pre-half-step v, not from v_tmp.'''
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70. * np.ones(num)
    m = m_e_inf(v)
    h = h_e_inf(v)
    n = n_e_inf(v)
    t = 0.

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 3))

    c, g_k, g_na, g_l = 1., 80., 100., 0.1
    v_k, v_na, v_l = -100., 50., -67.

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, t_old = v, h, n, t

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
        h_inc = (h_e_inf(v) - h) / tau_h_e(v)
        n_inc = (n_e_inf(v) - n) / tau_n_e(v)

        v_tmp = v + dt05_ * v_inc
        m_tmp = m_e_inf(v)  # faithful port of matlab's bug (uses v, not v_tmp)
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                  + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = (h_e_inf(v_tmp) - h_tmp) / tau_h_e(v_tmp)
        n_inc = (n_e_inf(v_tmp) - n_tmp) / tau_n_e(v_tmp)

        v = v + dt_ * v_inc
        m = m_e_inf(v)
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        t = t + dt_

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for k in ind:
            num_spikes[k] += 1
            if num_spikes[k] <= max_spikes:
                t_spikes[k, num_spikes[k] - 1] = (t_old * (-20 - v[k]) + t * (v_old[k] + 20)) / (v_old[k] - v[k])

        thr = t_spikes[:, max_spikes - 1] + phi_vec * (t_spikes[:, max_spikes - 1] - t_spikes[:, max_spikes - 2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    return out


def wb_init_population(i_ext, phi_vec):
    '''same as rtm_init_population but for the WB (I-cell) model.'''
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70. * np.ones(num)
    m = m_i_inf(v)
    h = h_i_inf(v)
    n = n_i_inf(v)
    t = 0.

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 3))

    c, g_k, g_na, g_l = 1., 9., 35., 0.1
    v_k, v_na, v_l = -90., 55., -65.

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, t_old = v, h, n, t

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
        h_inc = (h_i_inf(v) - h) / tau_h_i(v)
        n_inc = (n_i_inf(v) - n) / tau_n_i(v)

        v_tmp = v + dt05_ * v_inc
        m_tmp = m_i_inf(v)  # faithful port of matlab's bug (uses v, not v_tmp)
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                  + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = (h_i_inf(v_tmp) - h_tmp) / tau_h_i(v_tmp)
        n_inc = (n_i_inf(v_tmp) - n_tmp) / tau_n_i(v_tmp)

        v = v + dt_ * v_inc
        m = m_i_inf(v)
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        t = t + dt_

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for k in ind:
            num_spikes[k] += 1
            if num_spikes[k] <= max_spikes:
                t_spikes[k, num_spikes[k] - 1] = (t_old * (-20 - v[k]) + t * (v_old[k] + 20)) / (v_old[k] - v[k])

        thr = t_spikes[:, max_spikes - 1] + phi_vec * (t_spikes[:, max_spikes - 1] - t_spikes[:, max_spikes - 2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    return out


def spike_detection(t, v, threshold, dt):
    '''linear-interpolated up-crossing times of v through threshold.'''
    v = np.asarray(v)
    below = v[:-1] <= threshold
    above = v[1:] > threshold
    idx = np.where(below & above)[0]
    if len(idx) == 0:
        return np.empty(0)
    v0, v1 = v[idx], v[idx + 1]
    ts = (idx * dt * (v0 - threshold) + (idx + 1) * dt * (threshold - v1)) / (v0 - v1)
    return ts


def make_random_connectivity(num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii,
                              p_ee, p_ei, p_ie, p_ii, rng):
    '''each possible synapse exists independently with probability p_XY;
    weights are normalized by the expected in-degree so the expected total
    conductance onto a cell is g_hat_XY regardless of N or p.'''
    u_ee = rng.random((num_e, num_e))
    u_ei = rng.random((num_e, num_i))
    u_ie = rng.random((num_i, num_e))
    u_ii = rng.random((num_i, num_i))
    g_ee = g_hat_ee * (u_ee < p_ee) / (num_e * p_ee) if p_ee > 0 else np.zeros((num_e, num_e))
    g_ei = g_hat_ei * (u_ei < p_ei) / (num_e * p_ei)
    g_ie = g_hat_ie * (u_ie < p_ie) / (num_i * p_ie)
    g_ii = g_hat_ii * (u_ii < p_ii) / (num_i * p_ii)
    return g_ee, g_ei, g_ie, g_ii


def make_fixed_degree_connectivity(num_e, num_i, g_hat_ei, g_hat_ie, g_hat_ii,
                                    p_ei, p_ie, p_ii, rng, ni=1):
    '''each I-cell gets ni distinct, randomly-chosen presynaptic E-cells
    (feeding g_ei), each E-cell gets ni distinct presynaptic I-cells
    (feeding g_ie), each I-cell gets ni distinct presynaptic I-cells
    (feeding g_ii); g_ee stays all-zero.'''
    g_ee = np.zeros((num_e, num_e))
    g_ei = np.zeros((num_e, num_i))
    g_ie = np.zeros((num_i, num_e))
    g_ii = np.zeros((num_i, num_i))

    for j in range(num_i):
        i_vec = rng.choice(num_e, size=ni, replace=False)
        g_ei[i_vec, j] = g_hat_ei / (num_e * p_ei)
    for i in range(num_e):
        j_vec = rng.choice(num_i, size=ni, replace=False)
        g_ie[j_vec, i] = g_hat_ie / (num_i * p_ie)
    for j in range(num_i):
        j_vec = rng.choice(num_i, size=ni, replace=False)
        g_ii[j_vec, j] = g_hat_ii / (num_i * p_ii)

    return g_ee, g_ei, g_ie, g_ii


# ------------------------------------------------------- numba network stepper


@njit
def _m_e_inf_s(v):
    alpha_m = 0.32 * (v + 54) / (1 - math.exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (math.exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_e_inf_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_e_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


@njit
def _n_e_inf_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_e_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


@njit
def _m_i_inf_s(v):
    alpha_m = 0.1 * (v + 35) / (1 - math.exp(-(v + 35) / 10))
    beta_m = 4. * math.exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_i_inf_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_i_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5.


@njit
def _n_i_inf_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_i_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5.


@njit
def _ping_step_loop(m_steps, dt, dt05, num_e, num_i,
                     v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                     tau_r_i, tau_d_i, tau_dq_i,
                     i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
                     v_e, h_e, n_e, m_e, q_e, s_e,
                     v_i, h_i, n_i, m_i, q_i, s_i):
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)

    dve = np.empty(num_e); dne = np.empty(num_e); dhe = np.empty(num_e)
    dqe = np.empty(num_e); dse = np.empty(num_e)
    dvi = np.empty(num_i); dni = np.empty(num_i); dhi = np.empty(num_i)
    dqi = np.empty(num_i); dsi = np.empty(num_i)

    ve_m = np.empty(num_e); ne_m = np.empty(num_e); me_m = np.empty(num_e)
    he_m = np.empty(num_e); qe_m = np.empty(num_e); se_m = np.empty(num_e)
    vi_m = np.empty(num_i); ni_m = np.empty(num_i); mi_m = np.empty(num_i)
    hi_m = np.empty(num_i); qi_m = np.empty(num_i); si_m = np.empty(num_i)

    ve_old = np.empty(num_e)
    vi_old = np.empty(num_i)
    ee_term = np.empty(num_e); ie_term = np.empty(num_e)
    ei_term = np.empty(num_i); ii_term = np.empty(num_i)

    lfp = np.empty(m_steps + 1)
    lfp[0] = v_e.mean()

    for step in range(m_steps):
        k = step + 1

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * s_e[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * s_i[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * s_e[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * s_i[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = v_e[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(n_e[j], 4.0) * (-100 - v)
                      + 100 * math.pow(m_e[j], 3.0) * h_e[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - q_e[j]) / 0.1 - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e
        for j in range(num_i):
            v = v_i[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(n_i[j], 4.0) * (-90 - v)
                      + 35 * math.pow(m_i[j], 3.0) * h_i[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - q_i[j]) / 0.1 - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        for j in range(num_e):
            ve_m[j] = v_e[j] + dt05 * dve[j]
            ne_m[j] = n_e[j] + dt05 * dne[j]
            me_m[j] = _m_e_inf_s(ve_m[j])
            he_m[j] = h_e[j] + dt05 * dhe[j]
            qe_m[j] = q_e[j] + dt05 * dqe[j]
            se_m[j] = s_e[j] + dt05 * dse[j]
        for j in range(num_i):
            vi_m[j] = v_i[j] + dt05 * dvi[j]
            ni_m[j] = n_i[j] + dt05 * dni[j]
            mi_m[j] = _m_i_inf_s(vi_m[j])
            hi_m[j] = h_i[j] + dt05 * dhi[j]
            qi_m[j] = q_i[j] + dt05 * dqi[j]
            si_m[j] = s_i[j] + dt05 * dsi[j]

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * se_m[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * si_m[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * se_m[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * si_m[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = ve_m[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(ne_m[j], 4.0) * (-100 - v)
                      + 100 * math.pow(me_m[j], 3.0) * he_m[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - qe_m[j]) / 0.1 - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e
        for j in range(num_i):
            v = vi_m[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(ni_m[j], 4.0) * (-90 - v)
                      + 35 * math.pow(mi_m[j], 3.0) * hi_m[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - qi_m[j]) / 0.1 - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        for j in range(num_e):
            ve_old[j] = v_e[j]
        for j in range(num_i):
            vi_old[j] = v_i[j]

        for j in range(num_e):
            v_e[j] = v_e[j] + dt * dve[j]
            m_e[j] = _m_e_inf_s(v_e[j])
            h_e[j] = h_e[j] + dt * dhe[j]
            n_e[j] = n_e[j] + dt * dne[j]
            q_e[j] = q_e[j] + dt * dqe[j]
            s_e[j] = s_e[j] + dt * dse[j]
        for j in range(num_i):
            v_i[j] = v_i[j] + dt * dvi[j]
            m_i[j] = _m_i_inf_s(v_i[j])
            h_i[j] = h_i[j] + dt * dhi[j]
            n_i[j] = n_i[j] + dt * dni[j]
            q_i[j] = q_i[j] + dt * dqi[j]
            s_i[j] = s_i[j] + dt * dsi[j]

        for j in range(num_e):
            if ve_old[j] > -20 and v_e[j] <= -20:
                e_indices.append(j)
                e_times.append(((-20 - v_e[j]) * step * dt + (ve_old[j] + 20) * k * dt)
                                / (ve_old[j] - v_e[j]))
        for j in range(num_i):
            if vi_old[j] > -20 and v_i[j] <= -20:
                i_indices.append(j)
                i_times.append(((-20 - v_i[j]) * step * dt + (vi_old[j] + 20) * k * dt)
                                / (vi_old[j] - v_i[j]))

        lfp[k] = v_e.mean()

    return e_times, e_indices, i_times, i_indices, lfp


def simulate_ping_network(num_e, num_i, i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
                           v_rev_e=0., v_rev_i=-75.,
                           tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3.,
                           tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
                           t_final=200., dt=0.02, seed=63806, rng=None, i_init="constant"):
    '''shared, numba-accelerated PING network stepper, reused by PING_1
    through PING_9. i_init selects how the I-cell population is initialized:
    "constant" (all cells start at the WB resting potential) or "wb"
    (splay-initialized around the WB limit cycle via wb_init_population,
    used by PING_7 and PING_8).'''
    if rng is None:
        rng = np.random.default_rng(seed)
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    iv = rtm_init_population(i_ext_e, rng.random(num_e))
    v_e, h_e, n_e = iv[:, 0], iv[:, 1], iv[:, 2]
    m_e = m_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    if i_init == "wb":
        iv_i = wb_init_population(i_ext_i, rng.random(num_i))
        v_i, h_i, n_i = iv_i[:, 0], iv_i[:, 1], iv_i[:, 2]
        m_i = m_i_inf(v_i)
    else:
        v_i = -75. * np.ones(num_i)
        m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    t_e, i_e, t_i, i_i, lfp = _ping_step_loop(
        m_steps, dt, dt05, num_e, num_i,
        v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
        tau_r_i, tau_d_i, tau_dq_i,
        i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
        v_e, h_e, n_e, m_e, q_e, s_e,
        v_i, h_i, n_i, m_i, q_i, s_i,
    )
    t_e = np.array(t_e) if len(t_e) else np.empty(0)
    i_e = np.array(i_e, dtype=int) if len(i_e) else np.empty(0, dtype=int)
    t_i = np.array(t_i) if len(t_i) else np.empty(0)
    i_i = np.array(i_i, dtype=int) if len(i_i) else np.empty(0, dtype=int)
    return t_e, i_e, t_i, i_i, np.array(lfp)


def plot_ping_drive(t_e, i_e, t_i, i_i, lfp, num_e, num_i, t_final, title=""):
    '''raster (I cells below, E cells above a dashed separator) plus the
    LFP-like mean E-cell voltage; shared by PING_1..PING_4 and PING_7..PING_9.'''
    fig, axes = plt.subplots(2, 1, figsize=(8, 6))
    ax = axes[0]
    if len(t_i) > 0:
        ax.plot(t_i, i_i, '.b', markersize=2)
    if len(t_e) > 0:
        ax.plot(t_e, i_e + num_i, '.r', markersize=2)
    ax.plot([0, t_final], [num_i + 0.5, num_i + 0.5], '--k', linewidth=1)
    ax.set_yticks([num_i, num_e + num_i])
    ax.axis([0, t_final, 0, num_e + num_i + 1])
    if title:
        ax.set_title(title)

    lfp_t = np.linspace(0, t_final, len(lfp))
    axes[1].plot(lfp_t, lfp, '-k', linewidth=2)
    axes[1].set_xlabel('$t$ [ms]')
    axes[1].set_ylabel('mean($v$), E-cells')
    axes[1].axis([0, t_final, -100, 50])

    plt.tight_layout()
    return fig


def plot_ping_panels(panels, num_e, num_i, t_final, xlim=None):
    '''stack several (t_e, i_e, t_i, i_i, lfp) rasters, one per connectivity
    regime; shared by PING_5 and PING_6.'''
    fig, axes = plt.subplots(len(panels), 1, figsize=(8, 3 * len(panels)))
    if len(panels) == 1:
        axes = [axes]
    for ax, panel in zip(axes, panels):
        t_e, i_e, t_i, i_i = panel[0], panel[1], panel[2], panel[3]
        if len(t_i):
            ax.plot(t_i, i_i, '.b', markersize=2)
        if len(t_e):
            ax.plot(t_e, i_e + num_i, '.r', markersize=2)
        ax.plot([0, t_final], [num_i + 0.5, num_i + 0.5], '--k', linewidth=1)
        ax.set_yticks([num_i, num_e + num_i])
        ax.axis([*(xlim if xlim else (0, t_final)), 0, num_e + num_i + 1])
    axes[-1].set_xlabel('$t$ [ms]')
    plt.tight_layout()
    return fig

## 2-Cell PING: the minimal E-I loop

Two synaptically coupled cells -- one RTM (E), one WB (I) -- reproduce the
core PING mechanism: `simulate_2_cell_ping` integrates the pair with
`scipy.integrate.odeint` and reports the E-cell's oscillation period. The
E-to-I and I-to-E coupling strengths are equal by default (`g_ei=g_ie=0.25`);
try the slider below to see how the inhibitory feedback strength `g_ie`
shapes the period.

In [ ]:
def derivative_2cell(x0, t, i_ext_e, i_ext_i, g_ei, g_ie,
                      v_rev_e, v_rev_i,
                      tau_r_e, tau_d_e, tau_dq_e,
                      tau_r_i, tau_d_i, tau_dq_i):
    v_e, h_e, n_e, q_e, s_e, v_i, h_i, n_i, q_i, s_i = x0

    I_L_e = 0.1 * (v_e + 67.0)
    I_K_e = 80 * n_e ** 4 * (v_e + 100.0)
    I_Na_e = 100 * h_e * m_e_inf(v_e) ** 3 * (v_e - 50.0)
    I_syn_e = g_ie * s_i * (v_rev_i - v_e)

    dv_e = i_ext_e - I_L_e - I_K_e - I_Na_e + I_syn_e
    dh_e = (h_e_inf(v_e) - h_e) / tau_h_e(v_e)
    dn_e = (n_e_inf(v_e) - n_e) / tau_n_e(v_e)
    dq_e = 0.5 * (1 + np.tanh(0.1 * v_e)) * (1.0 - q_e) * 10.0 - q_e / tau_dq_e
    ds_e = q_e * (1.0 - s_e) / tau_r_e - s_e / tau_d_e

    I_L_i = 0.1 * (v_i + 65.0)
    I_K_i = 9.0 * n_i ** 4 * (v_i + 90.0)
    I_Na_i = 35.0 * m_i_inf(v_i) ** 3 * h_i * (v_i - 55.0)
    I_syn_i = g_ei * s_e * (v_rev_e - v_i)

    dv_i = i_ext_i - I_Na_i - I_K_i - I_L_i + I_syn_i
    dh_i = (h_i_inf(v_i) - h_i) / tau_h_i(v_i)
    dn_i = (n_i_inf(v_i) - n_i) / tau_n_i(v_i)
    dq_i = 0.5 * (1.0 + np.tanh(0.1 * v_i)) * (1.0 - q_i) * 10 - q_i / tau_dq_i
    ds_i = q_i * (1.0 - s_i) / tau_r_i - s_i / tau_d_i

    return np.array([dv_e, dh_e, dn_e, dq_e, ds_e,
                      dv_i, dh_i, dn_i, dq_i, ds_i])


def simulate_2_cell_ping(i_ext_e=1.4, i_ext_i=0.0, g_ei=0.25, g_ie=0.25,
                          tau_d_i=9.0, t_final=200.0, dt=0.01):
    v_rev_e, v_rev_i = 0.0, -75.0
    tau_r_e, tau_peak_e, tau_d_e = 0.5, 0.5, 3.0
    tau_r_i, tau_peak_i = 0.5, 0.5
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    x0 = [-75.0, 0.1, 0.1, 0.0, 0.0, -75.0, 0.1, 0.1, 0.0, 0.0]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative_2cell, x0, t,
                 args=(i_ext_e, i_ext_i, g_ei, g_ie, v_rev_e, v_rev_i,
                       tau_r_e, tau_d_e, tau_dq_e, tau_r_i, tau_d_i, tau_dq_i))
    v_e, v_i = sol[:, 0], sol[:, 5]
    e_spikes = spike_detection(t, v_e, -20.0, dt)
    return t, v_e, v_i, e_spikes


def plot_2_cell_ping(t, v_e, v_i, t_final):
    plt.figure(figsize=(7, 3))
    plt.plot(t, v_e, lw=2, c="r", label=r"$v_e$")
    plt.plot(t, v_i, lw=2, c="b", label=r"$v_i$")
    plt.xlim(min(t), max(t))
    plt.xlabel("time [ms]", fontsize=16)
    plt.ylabel("v [mV]", fontsize=16)
    plt.legend(fontsize=14, loc="upper right")
    plt.xticks(range(0, int(t_final) + 1, 50))
    plt.tick_params(labelsize=14)
    plt.tight_layout()

In [ ]:
t, v_e, v_i, e_spikes = simulate_2_cell_ping()
period = e_spikes[-1] - e_spikes[-2]
print("Period of E neuron %10.3f ms" % period)
plot_2_cell_ping(t, v_e, v_i, t_final=200.0)
plt.show()

In [ ]:
interact(lambda g_ie=0.25: plot_2_cell_ping(*simulate_2_cell_ping(g_ie=g_ie)[:3], t_final=200.0),
         g_ie=(0.05, 0.6, 0.05));

## 2-Cell PING: sensitivity of the period

`run_condition_numbers` perturbs the drive, the inhibitory coupling strength,
and the inhibitory decay time one at a time (each by about 1%) and reports
the percentage change in the E-cell's period relative to the unperturbed
baseline -- a quick way to see which parameter the rhythm is most sensitive
to.

In [ ]:
def run_condition_numbers():
    dt = 0.001
    _, _, _, base_spikes = simulate_2_cell_ping(dt=dt)
    base_period = base_spikes[-1] - base_spikes[-2]
    print("Period of E neuron %10.3f ms" % base_period)

    _, _, _, spikes_i_ext = simulate_2_cell_ping(i_ext_e=1.4 * 0.99, dt=dt)
    period_i_ext = spikes_i_ext[-1] - spikes_i_ext[-2]
    pct_i_ext = (base_period - period_i_ext) / base_period * 100
    print("Percentage change of reduce in I_E %10.3f" % pct_i_ext)

    _, _, _, spikes_g_ie = simulate_2_cell_ping(g_ie=0.25 * 1.01, dt=dt)
    period_g_ie = spikes_g_ie[-1] - spikes_g_ie[-2]
    pct_g_ie = (base_period - period_g_ie) / base_period * 100
    print("Percentage change of increse in g_IE %10.3f" % pct_g_ie)

    _, _, _, spikes_tau_i = simulate_2_cell_ping(tau_d_i=9.0 * 1.01, dt=dt)
    period_tau_i = spikes_tau_i[-1] - spikes_tau_i[-2]
    pct_tau_i = (base_period - period_tau_i) / base_period * 100
    print("Percentage change of increse in tau_I %10.3f" % pct_tau_i)

    return base_period, pct_i_ext, pct_g_ie, pct_tau_i

In [ ]:
base_period, pct_i_ext, pct_g_ie, pct_tau_i = run_condition_numbers()

## PING_1 - PING_4: heterogeneous and sparse populations

PING_1-PING_4 are all the same random E/I network, run with
`simulate_ping_network` (defined above), just with different heterogeneity
(`sigma_e`), connection probability (`p_XY`), and population size (`num_e`,
`num_i`):

- **PING_1**: heterogeneous drive (`sigma_e=0.05`), 50% connectivity.
- **PING_2**: homogeneous drive (`sigma_e=0`), all-to-all connectivity.
- **PING_3**: heterogeneous drive, sparse (5%) connectivity.
- **PING_4**: PING_3 scaled up to 800 E-cells / 200 I-cells.

In [ ]:
def simulate_ping_population(num_e=200, num_i=50, sigma_e=0.05, sigma_i=0.0,
                              g_hat_ee=0.0, g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25,
                              p_ee=0.5, p_ei=0.5, p_ie=0.5, p_ii=0.5,
                              i_ext_e_mean=1.4, i_ext_i_mean=0.0,
                              t_final=200.0, dt=0.02, seed=63806):
    rng = np.random.default_rng(seed)
    i_ext_e = i_ext_e_mean * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = i_ext_i_mean * np.ones(num_i) * (1 + sigma_i * rng.standard_normal(num_i))

    g_ee, g_ei, g_ie, g_ii = make_random_connectivity(
        num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii, p_ee, p_ei, p_ie, p_ii, rng)

    t_e, i_e, t_i, i_i, lfp = simulate_ping_network(
        num_e, num_i, i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
        t_final=t_final, dt=dt, rng=rng, i_init="constant")
    return t_e, i_e, t_i, i_i, lfp, num_e, num_i, t_final

In [ ]:
res1 = simulate_ping_population(num_e=200, num_i=50, sigma_e=0.05, p_ee=0.5, p_ei=0.5, p_ie=0.5, p_ii=0.5)
plot_ping_drive(*res1, title="PING_1")
plt.show()

In [ ]:
res2 = simulate_ping_population(num_e=200, num_i=50, sigma_e=0.0, p_ee=1.0, p_ei=1.0, p_ie=1.0, p_ii=1.0)
plot_ping_drive(*res2, title="PING_2")
plt.show()

In [ ]:
res3 = simulate_ping_population(num_e=200, num_i=50, sigma_e=0.05, p_ee=0.05, p_ei=0.05, p_ie=0.05, p_ii=0.05)
plot_ping_drive(*res3, title="PING_3")
plt.show()

In [ ]:
res4 = simulate_ping_population(num_e=800, num_i=200, sigma_e=0.05, p_ee=0.05, p_ei=0.05, p_ie=0.05, p_ii=0.05)
plot_ping_drive(*res4, title="PING_4")
plt.show()

PING_1 and PING_3 differ only in connectivity density `p` (0.5 vs 0.05);
the slider below sweeps `p` continuously (with `p_ee=p_ei=p_ie=p_ii=p`) to
show the population raster becoming less regular as the network sparsens.

In [ ]:
def simulate_and_plot_ping(p=0.5, num_e=200, num_i=50, sigma_e=0.05, t_final=200.0):
    res = simulate_ping_population(num_e=num_e, num_i=num_i, sigma_e=sigma_e,
                                    p_ee=p, p_ei=p, p_ie=p, p_ii=p, t_final=t_final)
    plot_ping_drive(*res, title=f"p = {p:.2f}")


interact(lambda p=0.5: simulate_and_plot_ping(p=p), p=(0.02, 1.0, 0.02));

## PING_5: three connectivity regimes

PING_5 runs `simulate_ping_network` three times with the same random
population but different connectivity: dense random (50%), sparse random
(in-degree matched), and sparse fixed-degree (exactly one presynaptic
partner per postsynaptic target). The dense network should show a visibly
more regular/periodic population rhythm than the two sparse ones.

In [ ]:
def run_ping5_panels(t_final=500., seed=63806):
    rng = np.random.default_rng(seed)
    num_e, num_i = 200, 50
    i_ext_e = 1.4 * np.ones(num_e)
    i_ext_i = 0.0 * np.ones(num_i)
    g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii = 0., 0.25, 0.25, 0.25
    p_ee, p_ei, p_ie, p_ii = 0.5, 0.5, 0.5, 0.5

    g1 = make_random_connectivity(num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii,
                                   p_ee, p_ei, p_ie, p_ii, rng)
    panel1 = simulate_ping_network(num_e, num_i, i_ext_e, i_ext_i, *g1, t_final=t_final, dt=0.01, rng=rng)

    p2_ee, p2_ei, p2_ie, p2_ii = 1 / num_e, 1 / num_e, 1 / num_i, 1 / num_i
    g2 = make_random_connectivity(num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii,
                                   p2_ee, p2_ei, p2_ie, p2_ii, rng)
    panel2 = simulate_ping_network(num_e, num_i, i_ext_e, i_ext_i, *g2, t_final=t_final, dt=0.01, rng=rng)

    g3 = make_fixed_degree_connectivity(num_e, num_i, g_hat_ei, g_hat_ie, g_hat_ii,
                                         p2_ei, p2_ie, p2_ii, rng, ni=1)
    panel3 = simulate_ping_network(num_e, num_i, i_ext_e, i_ext_i, *g3, t_final=t_final, dt=0.01, rng=rng)

    return panel1, panel2, panel3

In [ ]:
panel1, panel2, panel3 = run_ping5_panels()
t_e_spikes_1, i_e_spikes_1, t_i_spikes_1, i_i_spikes_1, lfp_1 = panel1
t_e_spikes_2, i_e_spikes_2, t_i_spikes_2, i_i_spikes_2, lfp_2 = panel2
t_e_spikes_3, i_e_spikes_3, t_i_spikes_3, i_i_spikes_3, lfp_3 = panel3
plot_ping_panels([panel1, panel2, panel3], num_e=200, num_i=50, t_final=500.)
plt.show()

## PING_6: same three panels, plain NumPy

`simulate_ping_network_plain` is the same Heun/midpoint PING stepper as
`simulate_ping_network` above, but written in plain NumPy instead of numba
-- useful as a readable reference and for sanity-checking the accelerated
version. `run_connectivity_panels` repeats PING_5's three connectivity
regimes (dense random, sparse random, sparse fixed-degree) with it, over a
longer run (`t_final=2000` ms by default).

In [ ]:
def simulate_ping_network_plain(num_e, num_i, i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
                                 v_rev_e=0., v_rev_i=-75.,
                                 tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3.,
                                 tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
                                 t_final=2000., dt=0.01, seed=63806, rng=None):
    '''plain-numpy sibling of simulate_ping_network, kept unaccelerated on
    purpose as a slower reference implementation to compare against.'''
    if rng is None:
        rng = np.random.default_rng(seed)
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    iv = rtm_init_population(i_ext_e, rng.random(num_e))
    v_e, h_e, n_e = iv[:, 0], iv[:, 1], iv[:, 2]
    m_e = m_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    v_i = -75. * np.ones(num_i)
    m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    t_e_spikes, i_e_spikes = [], []
    t_i_spikes, i_i_spikes = [], []

    for k in range(1, m_steps + 1):
        v_e_inc = (0.1 * (-67 - v_e) + 80 * n_e ** 4 * (-100 - v_e) + 100 * m_e ** 3 * h_e * (50 - v_e)
                   + (g_ee.T @ s_e) * (v_rev_e - v_e) + (g_ie.T @ s_i) * (v_rev_i - v_e) + i_ext_e)
        n_e_inc = (n_e_inf(v_e) - n_e) / tau_n_e(v_e)
        h_e_inc = (h_e_inf(v_e) - h_e) / tau_h_e(v_e)
        q_e_inc = (1 + tanh(v_e / 10)) / 2 * (1 - q_e) / 0.1 - q_e / tau_dq_e
        s_e_inc = q_e * (1 - s_e) / tau_r_e - s_e / tau_d_e

        v_i_inc = (0.1 * (-65 - v_i) + 9 * n_i ** 4 * (-90 - v_i) + 35 * m_i ** 3 * h_i * (55 - v_i)
                   + (g_ei.T @ s_e) * (v_rev_e - v_i) + (g_ii.T @ s_i) * (v_rev_i - v_i) + i_ext_i)
        n_i_inc = (n_i_inf(v_i) - n_i) / tau_n_i(v_i)
        h_i_inc = (h_i_inf(v_i) - h_i) / tau_h_i(v_i)
        q_i_inc = (1 + tanh(v_i / 10)) / 2 * (1 - q_i) / 0.1 - q_i / tau_dq_i
        s_i_inc = q_i * (1 - s_i) / tau_r_i - s_i / tau_d_i

        v_e_tmp = v_e + dt05 * v_e_inc
        n_e_tmp = n_e + dt05 * n_e_inc
        m_e_tmp = m_e_inf(v_e_tmp)
        h_e_tmp = h_e + dt05 * h_e_inc
        q_e_tmp = q_e + dt05 * q_e_inc
        s_e_tmp = s_e + dt05 * s_e_inc

        v_i_tmp = v_i + dt05 * v_i_inc
        n_i_tmp = n_i + dt05 * n_i_inc
        m_i_tmp = m_i_inf(v_i_tmp)
        h_i_tmp = h_i + dt05 * h_i_inc
        q_i_tmp = q_i + dt05 * q_i_inc
        s_i_tmp = s_i + dt05 * s_i_inc

        v_e_inc = (0.1 * (-67 - v_e_tmp) + 80 * n_e_tmp ** 4 * (-100 - v_e_tmp)
                   + 100 * m_e_tmp ** 3 * h_e_tmp * (50 - v_e_tmp)
                   + (g_ee.T @ s_e_tmp) * (v_rev_e - v_e_tmp) + (g_ie.T @ s_i_tmp) * (v_rev_i - v_e_tmp) + i_ext_e)
        n_e_inc = (n_e_inf(v_e_tmp) - n_e_tmp) / tau_n_e(v_e_tmp)
        h_e_inc = (h_e_inf(v_e_tmp) - h_e_tmp) / tau_h_e(v_e_tmp)
        q_e_inc = (1 + tanh(v_e_tmp / 10)) / 2 * (1 - q_e_tmp) / 0.1 - q_e_tmp / tau_dq_e
        s_e_inc = q_e_tmp * (1 - s_e_tmp) / tau_r_e - s_e_tmp / tau_d_e

        v_i_inc = (0.1 * (-65 - v_i_tmp) + 9 * n_i_tmp ** 4 * (-90 - v_i_tmp)
                   + 35 * m_i_tmp ** 3 * h_i_tmp * (55 - v_i_tmp)
                   + (g_ei.T @ s_e_tmp) * (v_rev_e - v_i_tmp) + (g_ii.T @ s_i_tmp) * (v_rev_i - v_i_tmp) + i_ext_i)
        n_i_inc = (n_i_inf(v_i_tmp) - n_i_tmp) / tau_n_i(v_i_tmp)
        h_i_inc = (h_i_inf(v_i_tmp) - h_i_tmp) / tau_h_i(v_i_tmp)
        q_i_inc = (1 + tanh(v_i_tmp / 10)) / 2 * (1 - q_i_tmp) / 0.1 - q_i_tmp / tau_dq_i
        s_i_inc = q_i_tmp * (1 - s_i_tmp) / tau_r_i - s_i_tmp / tau_d_i

        v_e_old, v_i_old = v_e, v_i

        v_e = v_e + dt * v_e_inc
        m_e = m_e_inf(v_e)
        h_e = h_e + dt * h_e_inc
        n_e = n_e + dt * n_e_inc
        q_e = q_e + dt * q_e_inc
        s_e = s_e + dt * s_e_inc

        v_i = v_i + dt * v_i_inc
        m_i = m_i_inf(v_i)
        h_i = h_i + dt * h_i_inc
        n_i = n_i + dt * n_i_inc
        q_i = q_i + dt * q_i_inc
        s_i = s_i + dt * s_i_inc

        which_e = np.where((v_e_old > -20) & (v_e <= -20))[0]
        which_i = np.where((v_i_old > -20) & (v_i <= -20))[0]
        if len(which_e) > 0:
            i_e_spikes.extend(which_e.tolist())
            t_e_spikes.extend((((-20 - v_e[which_e]) * (k - 1) * dt + (v_e_old[which_e] + 20) * k * dt)
                                / (-v_e[which_e] + v_e_old[which_e])).tolist())
        if len(which_i) > 0:
            i_i_spikes.extend(which_i.tolist())
            t_i_spikes.extend((((-20 - v_i[which_i]) * (k - 1) * dt + (v_i_old[which_i] + 20) * k * dt)
                                / (-v_i[which_i] + v_i_old[which_i])).tolist())

    return (np.array(t_e_spikes), np.array(i_e_spikes),
            np.array(t_i_spikes), np.array(i_i_spikes))


def run_connectivity_panels(t_final_run=2000.):
    rng = np.random.default_rng(63806)
    num_e, num_i = 200, 50
    i_ext_e = 1.4 * np.ones(num_e)
    i_ext_i = 0.0 * np.ones(num_i)
    g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii = 0., 0.25, 0.25, 0.25
    p_ee, p_ei, p_ie, p_ii = 0.5, 0.5, 0.5, 0.5

    panels = []
    g1 = make_random_connectivity(num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii,
                                   p_ee, p_ei, p_ie, p_ii, rng)
    panels.append(simulate_ping_network_plain(num_e, num_i, i_ext_e, i_ext_i, *g1,
                                               t_final=t_final_run, rng=rng))

    sparse = (1 / num_e, 1 / num_e, 1 / num_i, 1 / num_i)
    g2 = make_random_connectivity(num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii,
                                   sparse[0], sparse[1], sparse[2], sparse[3], rng)
    panels.append(simulate_ping_network_plain(num_e, num_i, i_ext_e, i_ext_i, *g2,
                                               t_final=t_final_run, rng=rng))

    g3 = make_fixed_degree_connectivity(num_e, num_i, g_hat_ei, g_hat_ie, g_hat_ii,
                                         sparse[1], sparse[2], sparse[3], rng, ni=1)
    panels.append(simulate_ping_network_plain(num_e, num_i, i_ext_e, i_ext_i, *g3,
                                               t_final=t_final_run, rng=rng))
    return panels


def main():
    t_final = 2000.
    num_e, num_i = 200, 50
    panels = run_connectivity_panels()

    fig, axes = plt.subplots(3, 1, figsize=(8, 9))
    for ax, (t_e, i_e, t_i, i_i) in zip(axes, panels):
        if len(t_i) > 0:
            ax.plot(t_i, i_i, '.b', markersize=2)
        if len(t_e) > 0:
            ax.plot(t_e, i_e + num_i, '.r', markersize=2)
        ax.plot([0, t_final], [num_i + 0.5, num_i + 0.5], '--k', linewidth=1)
        ax.set_yticks([num_i, num_e + num_i])
        ax.axis([t_final - 200, t_final, 0, num_e + num_i + 1])
    axes[-1].set_xlabel('$t$ [ms]')

    plt.tight_layout()
    plt.savefig("fig.png")
    return fig

Calling `main()` reproduces the reference figure (dense/sparse-random/
fixed-degree rasters over the last 200 ms of a 2-second run) and saves it to
`fig.png`.

In [ ]:
main()
plt.show()

## PING_7, PING_8, PING_9: drive-dependent population rasters + LFP

`simulate_ping_drive` is a small, generalized wrapper around the shared
`simulate_ping_network` that exposes the mean I-cell drive (`i_ext_i_mean`),
E-E coupling strength (`g_hat_ee`), and I-cell initialization mode as
parameters -- PING_7, PING_8, and PING_9 are just three presets of it:

- **PING_7**: moderate I-cell drive (`i_ext_i_mean=0.7`), no E-E coupling,
  I-cells splay-initialized (`i_init="wb"`).
- **PING_8**: PING_7 with stronger I-cell drive (`i_ext_i_mean=0.9`).
- **PING_9**: no extra I-cell drive, but recurrent E-E coupling added
  (`g_hat_ee=0.25`) and I-cells start at rest (`i_init="constant"`).

In [ ]:
def simulate_ping_drive(i_ext_i_mean=0.7, sigma_i=0.05, g_hat_ee=0., i_init="wb",
                         t_final=200., seed=63806):
    num_e, num_i = 200, 50
    rng = np.random.default_rng(seed)
    i_ext_e = 1.4 * np.ones(num_e) * (1 + 0.05 * rng.standard_normal(num_e))
    i_ext_i = i_ext_i_mean * np.ones(num_i) * (1 + sigma_i * rng.standard_normal(num_i))
    g_ee, g_ei, g_ie, g_ii = make_random_connectivity(
        num_e, num_i, g_hat_ee, 0.25, 0.25, 0.25, 0.5, 0.5, 0.5, 0.5, rng)
    t_e, i_e, t_i, i_i, lfp = simulate_ping_network(
        num_e, num_i, i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
        t_final=t_final, dt=0.01, rng=rng, i_init=i_init)
    return t_e, i_e, t_i, i_i, lfp, num_e, num_i, t_final


def simulate_ping_7(seed=63806, t_final=200.):
    return simulate_ping_drive(i_ext_i_mean=0.7, sigma_i=0.05, g_hat_ee=0.,
                                i_init="wb", t_final=t_final, seed=seed)


def simulate_ping_8(seed=63806, t_final=200.):
    return simulate_ping_drive(i_ext_i_mean=0.9, sigma_i=0.05, g_hat_ee=0.,
                                i_init="wb", t_final=t_final, seed=seed)


def simulate_ping_9(seed=63806, t_final=200.):
    return simulate_ping_drive(i_ext_i_mean=0.0, sigma_i=0.0, g_hat_ee=0.25,
                                i_init="constant", t_final=t_final, seed=seed)

In [ ]:
plot_ping_drive(*simulate_ping_7(), title="PING_7")
plt.show()

In [ ]:
plot_ping_drive(*simulate_ping_8(), title="PING_8")
plt.show()

In [ ]:
plot_ping_drive(*simulate_ping_9(), title="PING_9")
plt.show()

PING_7 and PING_8 differ only in the mean I-cell drive (0.7 vs 0.9); the
slider below sweeps `i_ext_i_mean` continuously to show I-cell participation
and the LFP amplitude changing with drive strength.

In [ ]:
interact(lambda i_ext_i_mean=0.7: plot_ping_drive(*simulate_ping_drive(i_ext_i_mean=i_ext_i_mean),
                                                     title=f"i_ext_i_mean = {i_ext_i_mean:.2f}"),
         i_ext_i_mean=(0.0, 1.2, 0.05));